In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from wise_ai.data_preprocessing import load_raw_data
from wise_ai.prediction_engine import predict_waste_risk
from wise_ai.recommendation_system import generate_recommendations
from wise_ai.dashboard import (
    get_overview_kpis,
    get_critical_products,
    get_branch_performance,
)
from wise_ai.alert_system import get_expiry_alerts
from wise_ai.analytics import get_stock_efficiency


Project root: c:\Users\COLLAPSION\Music\AI\TRIAL


In [4]:
df_raw = load_raw_data()  # pakai path default dari config
df_raw.head()


,Product_ID,Product_Name,Category,Initial_Stock,Sold_Quantity,Remaining_Stock,Expiry_Days_Left,Price,Discount_Applied,Temperature_(°C),Weather,Day_Type,Branch,Historical_Avg_Sales,Waste_Amount,Waste_Flag
0,PRD001,Bakery Product 1,Bakery,111,92,19,5,57073,15,26.7,Sunny,Holiday,Semarang,56,13,1
1,PRD002,Dairy Product 2,Dairy,185,86,99,2,8121,5,26.2,Rainy,Holiday,Yogyakarta,53,85,1
2,PRD003,Fruit Product 3,Fruit,116,41,75,7,40352,25,26.3,Sunny,Weekday,Semarang,107,46,1
3,PRD004,Bakery Product 4,Bakery,133,111,22,3,33857,5,30.1,Cloudy,Weekend,Bandung,59,4,1
4,PRD005,Fruit Product 5,Fruit,69,37,32,5,31406,0,27.5,Cloudy,Holiday,Semarang,48,14,1


In [9]:
df_with_risk = predict_waste_risk(df_raw)
df_with_risk.head()



,Product_ID,Product_Name,Category,Initial_Stock,Sold_Quantity,Remaining_Stock,Expiry_Days_Left,Price,Discount_Applied,Temperature_(°C),Weather,Day_Type,Branch,Historical_Avg_Sales,Waste_Amount,Waste_Flag,waste_proba,risk_level
0,PRD001,Bakery Product 1,Bakery,111,92,19,5,57073,15,26.7,Sunny,Holiday,Semarang,56,13,1,0.985,High
1,PRD002,Dairy Product 2,Dairy,185,86,99,2,8121,5,26.2,Rainy,Holiday,Yogyakarta,53,85,1,0.970,High
2,PRD003,Fruit Product 3,Fruit,116,41,75,7,40352,25,26.3,Sunny,Weekday,Semarang,107,46,1,0.980,High
3,PRD004,Bakery Product 4,Bakery,133,111,22,3,33857,5,30.1,Cloudy,Weekend,Bandung,59,4,1,0.960,High
4,PRD005,Fruit Product 5,Fruit,69,37,32,5,31406,0,27.5,Cloudy,Holiday,Semarang,48,14,1,0.980,High


In [10]:
df_with_risk["risk_level"].value_counts()

risk_level
High      875
Low        95
Medium     30
Name: count, dtype: int64

In [11]:
df_recommended = generate_recommendations(df_with_risk)
df_recommended[[
    "Product_ID",
    "Product_Name",
    "risk_level",
    "waste_proba",
    "Remaining_Stock",
    "Expiry_Days_Left",
    "action_discount",
    "action_redistribute",
    "action_donate",
    "action_recipe",
]].head(10)


,Product_ID,Product_Name,risk_level,waste_proba,Remaining_Stock,Expiry_Days_Left,action_discount,action_redistribute,action_donate,action_recipe
0,PRD001,Bakery Product 1,High,0.985,19,5,30,False,False,True
1,PRD002,Dairy Product 2,High,0.970,99,2,30,True,True,False
2,PRD003,Fruit Product 3,High,0.980,75,7,30,True,False,True
3,PRD004,Bakery Product 4,High,0.960,22,3,30,False,False,True
4,PRD005,Fruit Product 5,High,0.980,32,5,30,False,False,True
5,PRD006,Fruit Product 6,High,0.920,101,4,30,True,False,True
6,PRD007,Fruit Product 7,High,0.960,10,7,30,False,False,True
7,PRD008,Dairy Product 8,High,0.975,9,4,30,False,False,False
8,PRD009,Meat Product 9,High,1.000,41,5,30,False,False,False
9,PRD010,Vegetable Product 10,High,1.000,68,5,30,True,False,False


In [12]:
kpis = get_overview_kpis(df_recommended)
kpis


{'total_products': 1000,
 'high_risk_products': 875,
 'total_waste_amount': 23604.0}

In [13]:
critical_products = get_critical_products(df_recommended, top_n=10)
critical_products[[
    "Product_ID",
    "Product_Name",
    "Branch",
    "risk_level",
    "waste_proba",
    "Remaining_Stock",
    "Expiry_Days_Left"
]]


,Product_ID,Product_Name,Branch,risk_level,waste_proba,Remaining_Stock,Expiry_Days_Left
482,PRD1182,Fruit Product 216 Extra182,Bandung,High,1.0,69,3
119,PRD120,Fruit Product 120,Bandung,High,1.0,113,7
524,PRD1224,Bakery Product 228 Extra224,Medan,High,1.0,68,3
560,PRD1260,Dairy Product 47 Extra260,Surabaya,High,1.0,44,6
574,PRD1274,Fruit Product 159 Extra274,Semarang,High,1.0,54,5
183,PRD184,Bakery Product 184,Surabaya,High,1.0,41,4
151,PRD152,Meat Product 152,Yogyakarta,High,1.0,68,7
147,PRD148,Fruit Product 148,Yogyakarta,High,1.0,67,2
142,PRD143,Bakery Product 143,Medan,High,1.0,89,1
135,PRD136,Fruit Product 136,Bandung,High,1.0,49,2


In [14]:
alerts = get_expiry_alerts(df_recommended)
alerts[[
    "Product_ID",
    "Product_Name",
    "Expiry_Days_Left",
    "risk_level",
    "waste_proba",
    "Remaining_Stock"
]].head(15)


,Product_ID,Product_Name,Expiry_Days_Left,risk_level,waste_proba,Remaining_Stock
397,PRD1097,Beverage Product 101 Extra97,0,High,1.000,44
317,PRD1017,Beverage Product 150 Extra17,0,High,0.995,89
372,PRD1072,Beverage Product 252 Extra72,0,High,0.995,127
969,PRD1669,Dairy Product 279 Extra669,0,High,0.995,41
567,PRD1267,Vegetable Product 298 Extra267,0,High,0.990,54
759,PRD1459,Meat Product 23 Extra459,0,High,0.990,132
762,PRD1462,Beverage Product 176 Extra462,0,High,0.990,46
821,PRD1521,Vegetable Product 298 Extra521,0,High,0.990,65
459,PRD1159,Dairy Product 187 Extra159,0,High,0.985,67
465,PRD1165,Vegetable Product 32 Extra165,0,High,0.985,93


In [16]:
stock_eff.sort_values("avg_waste_proba", ascending=False)


,Category,avg_remaining_stock,avg_waste_proba
3,Fruit,52.294521,0.897945
5,Vegetable,39.641618,0.891012
1,Beverage,51.512346,0.887469
0,Bakery,46.245810,0.873994
2,Dairy,39.792135,0.863820
4,Meat,43.148148,0.838395


In [18]:
processed_path = PROJECT_ROOT / "data" / "processed" / "dataset_with_risk_reco.xlsx"
processed_path.parent.mkdir(parents=True, exist_ok=True)

df_recommended.to_excel(processed_path, index=False)
processed_path


WindowsPath('c:/Users/COLLAPSION/Music/AI/TRIAL/data/processed/dataset_with_risk_reco.xlsx')